In [ ]:
# imports
import csv
from pathlib import Path
import pandas as pd
import numpy as np
import os
from scipy.stats import spearmanr, ttest_rel, pearsonr
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
import math
import pingouin as pg
import random
from datetime import datetime
from scipy.signal import resample
from scipy.interpolate import interp1d
import matplotlib as mpl

# Ensures consistent PDF output formatting (vector fonts)
mpl.rcParams["ps.useafm"] = True


In [ ]:
# Gets all day directories (like D01, D02, etc.) for a specific subject.
# Filters folder names that begin with "D", sorts them, and returns the list of paths.
def get_day_paths(subject_path: Path): # checked
    days = os.listdir(subject_path)
    day_paths = sorted(list(map(lambda x: subject_path / x, list(filter(lambda x: x[0] == "D", days)))))

    return day_paths

In [ ]:
# Reads the processed.csv file inside the given day/segment folder 
# Returns a dictionary of DataFrames with keys: "affected finger", "affected wrist", "less affected finger", "less affected wrist
def read_processed_sensor_csv(day_path: Path): # checked
    """
    Read all sensor data. Concatenate and order if there are more than one file.
    """
    assigned_data: pd.DataFrame = pd.read_csv(list(day_path.glob("processed.csv"))[0])

    affected_finger: pd.DataFrame = assigned_data["affected finger"]
    affected_wrist: pd.DataFrame = assigned_data["affected wrist"]
    less_affected_finger: pd.DataFrame = assigned_data["less affected finger"]
    less_affected_wrist: pd.DataFrame = assigned_data["less affected wrist"]

    return {"affected finger": affected_finger,
            "affected wrist": affected_wrist,
            "less affected finger": less_affected_finger,
            "less affected wrist": less_affected_wrist}

In [ ]:
# Gets all subject directories inside the main data folder, ignoring hidden files
# For each subject folder:
    # Calls get_day_paths() to get all day paths.
    # Stores them in a dictionary where the key is the subject name and the value is a list of day paths.
def get_subject_day_paths(data_path: Path) -> dict[str, list[Path]]: # checked
    all_subject_dir_names: list[str] = os.listdir(data_path)
    all_subject_paths: list[Path] = sorted(list(
        map(lambda x: data_path / x, list(filter(lambda x: x[0] != ".", all_subject_dir_names)))
    ))
    subject_day_paths = {}

    for subject_path in all_subject_paths:
        subject = subject_path.name
        subject_day_paths[subject] = get_day_paths(subject_path=subject_path)

    return subject_day_paths

In [ ]:
# Prepares the output folder structure to store only subject-level directories (used later when combining across days).
def create_subject_dirs(valid_subject_day_paths: dict[str, list[Path]], output_dir: Path): # checked
    subjects: list[str] = valid_subject_day_paths.keys()

    for subject in subjects:
        subject_output_path: Path = output_dir / subject
        if not os.path.exists(subject_output_path):
            os.makedirs(subject_output_path)

In [ ]:
def combine_data(ready_dir: Path, output_dir: Path):
    """
    Combine all processed daily data for each subject into a single CSV.

    For each subject:
        - Reads each day's 'processed.csv'
        - Concatenates across days
        - Saves one 'combined.csv' per subject

    Args:
        ready_dir (Path): Directory containing processed day folders.
        output_dir (Path): Directory where combined data will be stored.
    """
    data_types = ["wrist", "finger"]
    valid_subject_day_paths: dict[str, list[Path]] = get_subject_day_paths(data_path=ready_dir)
    create_subject_dirs(valid_subject_day_paths=valid_subject_day_paths, output_dir=output_dir)

    for subject, day_paths in valid_subject_day_paths.items():
        combined_dict: dict[str, np.ndarray] = {
            "affected finger": np.array([]),
            "affected wrist": np.array([]),
            "less affected finger": np.array([]),
            "less affected wrist": np.array([])
        }

        for day_path in day_paths:
            print(subject, day_path.name)
            data_day_dict: dict[str, pd.DataFrame] = read_processed_sensor_csv(day_path)

            # Concatenate data across days
            for data_type in data_types:
                combined_dict[f"affected {data_type}"] = np.concatenate(
                    (combined_dict[f"affected {data_type}"], data_day_dict[f"affected {data_type}"]),
                    axis=0
                )
                combined_dict[f"less affected {data_type}"] = np.concatenate(
                    (combined_dict[f"less affected {data_type}"], data_day_dict[f"less affected {data_type}"]),
                    axis=0
                )

        # Save combined subject-level data
        df_combined = pd.DataFrame({
            "affected finger": pd.Series(combined_dict["affected finger"]),
            "affected wrist": pd.Series(combined_dict["affected wrist"]),
            "less affected finger": pd.Series(combined_dict["less affected finger"]),
            "less affected wrist": pd.Series(combined_dict["less affected wrist"])
        })
        
        df_combined.to_csv(output_dir / subject / f"combined.csv", index=False)

In [ ]:
combine_data(ready_dir=Path("path/to/your/data"), output_dir=Path("path/to/your/data"))
